In [ ]:
# ============================================================
# GIÁO TRÌNH: THỊ GIÁC MÁY: TỪ XỬ LÝ ẢNH ĐẾN HỌC SÂU
# BÀI CODE MINH HỌA: SO SÁNH CÁC BỘ TỐI ƯU TRÊN TINYPLAIN-56
# Chương/Mục liên quan: Chương 7 - Huấn luyện mô hình học sâu
# ============================================================

# ============================================================
# MÔ TẢ
# ============================================================

# Mục đích:
# - Minh họa ảnh hưởng của các bộ tối ưu GD, SGD, Momentum, Adam và AdamW.
# - Quan sát loss, accuracy, generalization gap và gradient norm trong quá trình huấn luyện.
# - Liên hệ quá trình cập nhật trọng số với thuật toán lan truyền ngược và tối ưu hóa gradient.

# Input:
# - Dataset CIFAR-10.
# - Kiểu dữ liệu: ảnh màu RGB kích thước 32 x 32.

# Output:
# - Đường cong train loss, test loss.
# - Đường cong train accuracy, test accuracy.
# - Đường cong generalization gap.
# - Đường cong gradient norm.
# - Bảng tổng kết kết quả cuối cùng của từng optimizer.

# Lưu ý:
# Đoạn code này được xây dựng với sự hỗ trợ của công cụ AI.
# Giảng viên đã đọc, kiểm tra và hiệu chỉnh nhằm bảo đảm tính chính xác,
# tính sư phạm và sự phù hợp với nội dung lý thuyết trong giáo trình.

# ============================================================
# 1. CÀI ĐẶT VÀ IMPORT THƯ VIỆN
# ============================================================

import gc
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

# ============================================================
# 2. CẤU HÌNH THÍ NGHIỆM
# ============================================================

SEED = 42
STUDENT_MODE = False

if STUDENT_MODE:
    NUM_EPOCHS = 12
    TRAIN_SIZE = 5000
    TEST_SIZE = 1000
else:
    NUM_EPOCHS = 80
    TRAIN_SIZE = 10000
    TEST_SIZE = 2000

BATCH_SIZE = 32
TEST_BATCH_SIZE = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# ============================================================
# 3. TẢI DỮ LIỆU ĐẦU VÀO
# ============================================================

transform_train = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

train_full = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

test_full = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

train_set = Subset(train_full, list(range(TRAIN_SIZE)))
test_set = Subset(test_full, list(range(TEST_SIZE)))

# ============================================================
# 4. ĐỊNH NGHĨA MÔ HÌNH TINYPLAIN-56
# ============================================================

class PlainBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x


class TinyPlain(nn.Module):
    # TinyPlain-56 có độ sâu 6n + 2 với n = 9.
    # Kiến trúc tương tự ResNet cho CIFAR-10 nhưng không dùng skip connection.

    def __init__(self, num_classes=10, n=9):
        super().__init__()

        self.in_channels = 16

        self.conv1 = nn.Conv2d(
            3,
            16,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(16)

        self.stage1 = self._make_stage(16, n, stride=1)
        self.stage2 = self._make_stage(32, n, stride=2)
        self.stage3 = self._make_stage(64, n, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def _make_stage(self, out_channels, num_blocks, stride):
        layers = []

        layers.append(
            PlainBlock(self.in_channels, out_channels, stride=stride)
        )
        self.in_channels = out_channels

        for _ in range(1, num_blocks):
            layers.append(
                PlainBlock(out_channels, out_channels, stride=1)
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))

        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


def make_model():
    return TinyPlain(num_classes=10, n=9).to(DEVICE)

# ============================================================
# 5. ĐỊNH NGHĨA CÁC BỘ TỐI ƯU
# ============================================================

def make_optimizer(name, model):
    if name == "GD":
        return optim.SGD(model.parameters(), lr=0.05)

    if name == "SGD":
        return optim.SGD(model.parameters(), lr=0.05)

    if name == "SGD_Momentum":
        return optim.SGD(model.parameters(), lr=0.05, momentum=0.9)

    if name == "Adam":
        return optim.Adam(model.parameters(), lr=1e-3)

    if name == "AdamW":
        return optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

    raise ValueError(f"Unknown optimizer: {name}")

# ============================================================
# 6. HÀM ĐÁNH GIÁ VÀ TÍNH GRADIENT NORM
# ============================================================

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += x.size(0)

    return total_loss / total_samples, total_correct / total_samples


def compute_grad_norm(model):
    total_norm_sq = 0.0

    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.detach().data.norm(2)
            total_norm_sq += param_norm.item() ** 2

    return total_norm_sq ** 0.5

# ============================================================
# 7. HUẤN LUYỆN MÔ HÌNH VỚI MỘT OPTIMIZER
# ============================================================

def train_one_optimizer(opt_name):
    print(f"\n========== Training with {opt_name} ==========")

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    set_seed(SEED)

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )

    test_loader = DataLoader(
        test_set,
        batch_size=TEST_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    model = make_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(opt_name, model)

    history = {
        "optimizer": [],
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
        "generalization_gap": [],
        "grad_norm": [],
        "time_sec": [],
    }

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()

        start_time = time.time()

        running_loss = 0.0
        running_correct = 0
        running_samples = 0
        grad_norms = []

        if opt_name == "GD":
            optimizer.zero_grad()

            num_batches = len(train_loader)

            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)

                logits = model(x)
                loss = criterion(logits, y)

                loss_for_backward = loss / num_batches
                loss_for_backward.backward()

                running_loss += loss.item() * x.size(0)
                running_correct += (logits.argmax(dim=1) == y).sum().item()
                running_samples += x.size(0)

                del x, y, logits, loss, loss_for_backward

            grad_norms.append(compute_grad_norm(model))
            optimizer.step()

        else:
            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)

                optimizer.zero_grad()

                logits = model(x)
                loss = criterion(logits, y)

                loss.backward()

                grad_norms.append(compute_grad_norm(model))
                optimizer.step()

                running_loss += loss.item() * x.size(0)
                running_correct += (logits.argmax(dim=1) == y).sum().item()
                running_samples += x.size(0)

                del x, y, logits, loss

        train_loss = running_loss / running_samples
        train_acc = running_correct / running_samples

        test_loss, test_acc = evaluate(model, test_loader, criterion)

        gap = train_acc - test_acc
        avg_grad_norm = float(np.mean(grad_norms))
        elapsed = time.time() - start_time

        history["optimizer"].append(opt_name)
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        history["generalization_gap"].append(gap)
        history["grad_norm"].append(avg_grad_norm)
        history["time_sec"].append(elapsed)

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.3f} | "
            f"gap={gap:.3f} | grad_norm={avg_grad_norm:.3f}"
        )

    del model, optimizer, criterion
    gc.collect()

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return pd.DataFrame(history)

# ============================================================
# 8. CHẠY THÍ NGHIỆM
# ============================================================

optimizers_to_compare = [
    "GD",
    "SGD",
    "SGD_Momentum",
    "Adam",
    "AdamW",
]

all_histories = []

for opt_name in optimizers_to_compare:
    hist = train_one_optimizer(opt_name)
    all_histories.append(hist)

results = pd.concat(all_histories, ignore_index=True)
results.to_csv("tinyplain56_optimizer_comparison.csv", index=False)

print("\nSaved results to tinyplain56_optimizer_comparison.csv")
display(results.tail())

# ============================================================
# 9. HIỂN THỊ KẾT QUẢ
# ============================================================

def plot_metric(df, metric, ylabel, title):
    plt.figure(figsize=(8, 5))

    for opt_name in df["optimizer"].unique():
        sub = df[df["optimizer"] == opt_name]
        plt.plot(sub["epoch"], sub[metric], marker="o", label=opt_name)

    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_metric(
    results,
    metric="train_loss",
    ylabel="Training loss",
    title="TinyPlain-56: Training loss"
)

plot_metric(
    results,
    metric="test_loss",
    ylabel="Validation/Test loss",
    title="TinyPlain-56: Validation/Test loss"
)

plot_metric(
    results,
    metric="train_acc",
    ylabel="Training accuracy",
    title="TinyPlain-56: Training accuracy"
)

plot_metric(
    results,
    metric="test_acc",
    ylabel="Validation/Test accuracy",
    title="TinyPlain-56: Validation/Test accuracy"
)

plot_metric(
    results,
    metric="generalization_gap",
    ylabel="Train accuracy - test accuracy",
    title="TinyPlain-56: Generalization gap"
)

plot_metric(
    results,
    metric="grad_norm",
    ylabel="Average gradient norm",
    title="TinyPlain-56: Average gradient norm per epoch"
)

# ============================================================
# 10. KIỂM TRA KẾT QUẢ
# ============================================================

summary = (
    results
    .sort_values("epoch")
    .groupby("optimizer")
    .tail(1)
    [[
        "optimizer",
        "train_loss",
        "train_acc",
        "test_loss",
        "test_acc",
        "generalization_gap",
        "grad_norm"
    ]]
    .sort_values("test_acc", ascending=False)
)

display(summary)

assert len(results) == NUM_EPOCHS * len(optimizers_to_compare)
assert set(results["optimizer"].unique()) == set(optimizers_to_compare)
assert results[["train_loss", "test_loss", "train_acc", "test_acc"]].notnull().all().all()

print("Kiểm tra hoàn tất: kết quả có đủ số epoch, đủ optimizer và không có giá trị rỗng.")

# ============================================================
# 11. GỢI Ý THỬ NGHIỆM CHO NGƯỜI HỌC
# ============================================================

# 1. Đổi STUDENT_MODE = True để chạy nhanh hơn trên Colab.
# 2. Thay đổi NUM_EPOCHS để quan sát đường cong hội tụ dài hơn hoặc ngắn hơn.
# 3. Thay đổi learning rate trong make_optimizer để quan sát ảnh hưởng đến loss và accuracy.

Device: cuda


100%|██████████| 170M/170M [00:22<00:00, 7.49MB/s]



========== Training with GD ==========
Epoch 01/80 | train_loss=2.3500, train_acc=0.090 | test_loss=681070.8530, test_acc=0.098 | gap=-0.007 | grad_norm=101.859
Epoch 02/80 | train_loss=2.3139, train_acc=0.101 | test_loss=81.6294, test_acc=0.099 | gap=0.002 | grad_norm=18.193
Epoch 03/80 | train_loss=2.3204, train_acc=0.096 | test_loss=28.8702, test_acc=0.101 | gap=-0.005 | grad_norm=11.211
Epoch 04/80 | train_loss=2.3142, train_acc=0.108 | test_loss=2.8708, test_acc=0.112 | gap=-0.003 | grad_norm=8.290
Epoch 05/80 | train_loss=2.2993, train_acc=0.102 | test_loss=4.1325, test_acc=0.127 | gap=-0.025 | grad_norm=6.202
Epoch 06/80 | train_loss=2.2846, train_acc=0.104 | test_loss=2.2751, test_acc=0.114 | gap=-0.010 | grad_norm=5.051
Epoch 07/80 | train_loss=2.2757, train_acc=0.108 | test_loss=9.0861, test_acc=0.128 | gap=-0.020 | grad_norm=5.734
Epoch 08/80 | train_loss=2.2835, train_acc=0.133 | test_loss=2.2520, test_acc=0.155 | gap=-0.022 | grad_norm=3.960
Epoch 09/80 | train_loss=2.266